# Example pipeline using TCRAFT Overhang Gen to design overhangs

In [ ]:
#prereq: install biopython, pandas, matplotlib, joblib
! pip install biopython pandas matplotlib joblib

In [1]:
import pandas as pd
from Bio.Seq import Seq
from TCRAFT_Overhang_Generator import OverhangSet, ScoringFunction, MCMC_Optimizer

In [ ]:
full_trac_seq = str(Seq(open('../path/to/Constant/Region/AminoAcid_Sequence.txt', 'r').read().strip()).translate())
trac_id = 'Constant Region ID'

all_trbvs = pd.read_csv('../path/to/Variable/Region/AminoAcid_Sequences.csv')
full_trbv_seqs = all_trbvs.Amino_Acid.tolist()
trbv_ids = all_trbvs.TRBV_ID.tolist()

oh_set = OverhangSet.build(full_trbv_seqs, 
                            full_trac_seq, 
                            trbv_ids,
                            trac_id,
                            restriction_enzyme='BsmBI', #pick between BsaI, BsmBI, and BbsI
                            order=OverhangSet.order_VLeft_CRight,
                            v_window=7, 
                            c_window=9,
                            banned_OHs=[])
score_fxn = ScoringFunction('./NEB_Ligation_Fidelity_Dataset', weight_coef=0.5)
optimizer = MCMC_Optimizer(oh_set, 'Overhang_Design_Example', score_fxn)

In [ ]:
# Optimization Run 1 - starting from 16 random start seeds.
# Running at a low temperature to focus on finding a good local optimum.
optimizer.optimize(temp=1e-4, num_iter=10000, random_init=True, n_seeds=16, n_processes=4)

In [ ]:
# Optimization Run 2 - starting from best Overhang Set result found in Optimization Run 1.
# Running at a higher temperature to explore more of the solution space.
optimizer.optimize(temp=1e-3, num_iter=5000, random_init=False, n_seeds=8, n_processes=4)

In [ ]:
# Plot the trace of scores over all iterations and optimization runs.
optimizer.plot_trace()

In [ ]:
# Save the final Overhang Set design and log files to the file path specified.
optimizer.save('./Path/to/Save/Directory')